# Market calibration: can a model beat the crowd?

Midterm, market part. Question: Polymarket prices are the crowd's probability that an event happens.
Are they calibrated in our four categories, and can a model built on information available at the time do better than the price?

Sample: resolved Yes/No markets in Politics, Technology, Entertainment and Economy (Polymarket's own event tags),
closing between Jun 1 and Oct 25, 2026. Price = last trade on the Yes side h hours before the market closed
(No trades converted with 1 - price). Rebuild the sample with `python scripts/build_market_sample.py`.

In [1]:
import sys, warnings; warnings.filterwarnings('ignore')
sys.path.insert(0, '../src')
import numpy as np, pandas as pd, statsmodels.api as sm
from scipy.special import logit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, log_loss
pd.set_option('display.width', 200)
rng = np.random.default_rng(0)

s = pd.read_parquet('../data/processed/market_sample.parquet')
d = s[s.has_price].copy().reset_index(drop=True)     # a trade within 72h of the horizon
d['lp'] = logit(d.p_last.clip(0.005, 0.995))
d.groupby('horizon_h').agg(markets=('slug','size'), events=('event_slug','nunique'), yes_rate=('y','mean')).round(3)

,markets,events,yes_rate
horizon_h,,,
6,5919,1628,0.211
24,5805,1610,0.202
72,5645,1561,0.191
168,4178,1265,0.199


## 1. Calibration

Calibration slope: logistic regression of the outcome on logit(price). Slope 1 = calibrated, above 1 = prices too close to 50/50
(longshots overpriced, favorites underpriced, the favorite-longshot bias), below 1 = overconfident.
Confidence intervals resample whole events, because markets in one event are not independent.

In [2]:
def slope(g):
    m = sm.GLM(g.y, sm.add_constant(g[['lp']]), family=sm.families.Binomial()).fit(
        cov_type='cluster', cov_kwds={'groups': pd.factorize(g.event_slug)[0]})
    return pd.Series({'n': len(g), 'events': g.event_slug.nunique(), 'slope': m.params['lp'],
                      'lo': m.params['lp'] - 1.96*m.bse['lp'], 'hi': m.params['lp'] + 1.96*m.bse['lp'],
                      'brier': ((g.p_last - g.y)**2).mean()})

rows = []
for h in [6, 24, 72, 168]:
    for cat in ['All', 'Politics', 'Technology', 'Entertainment', 'Economy']:
        g = d[(d.horizon_h == h) & ((d.category == cat) | (cat == 'All'))]
        rows.append(slope(g).rename((h, cat)))
pd.DataFrame(rows).round(3)

n  events  slope     lo     hi  brier
6   All            5919.0  1628.0  1.078  1.001  1.155  0.038
    Politics       2542.0   855.0  1.069  0.950  1.187  0.028
    Technology      740.0   210.0  1.078  0.851  1.305  0.031
    Entertainment  1875.0   411.0  1.169  1.028  1.310  0.047
    Economy         762.0   152.0  0.929  0.753  1.105  0.059
24  All            5805.0  1610.0  1.050  0.980  1.121  0.049
    Politics       2574.0   856.0  1.034  0.929  1.139  0.043
    Technology      729.0   203.0  1.111  0.947  1.274  0.038
    Entertainment  1753.0   399.0  1.124  0.974  1.273  0.054
    Economy         749.0   152.0  0.917  0.740  1.093  0.068
72  All            5645.0  1561.0  1.036  0.964  1.108  0.057
    Politics       2630.0   847.0  1.002  0.901  1.102  0.056
    Technology      682.0   195.0  1.182  0.980  1.385  0.043
    Entertainment  1632.0   369.0  1.057  0.912  1.202  0.060
    Economy         701.0   150.0  1.008  0.797  1.219  0.069
168 All            4178.0  1265.0  0.949  0.877  1.021  0.074
    Politics       2216.0   775.0  0.941  0.841  1.041  0.078
    Technology      456.0   130.0  0.909  0.707  1.110  0.061
    Entertainment   856.0   213.0  0.977  0.823  1.130  0.070
    Economy         650.0   147.0  0.986  0.789  1.183  0.077

Robustness: where does the slope above 1 come from? Drop the near-certain prices (below 2c or above 98c), and look at it month by month.

In [3]:
for h in [6, 24]:
    g = d[(d.horizon_h == h) & d.p_last.between(0.02, 0.98)]
    print(h, 'h, prices 2c to 98c:', slope(g).round(3).to_dict())
d[d.horizon_h == 24].groupby(d.resolved_at.dt.to_period('M')).apply(slope).round(3)

6 h, prices 2c to 98c: {'n': 1750.0, 'events': 729.0, 'slope': 1.002, 'lo': 0.894, 'hi': 1.109, 'brier': 0.123}
24 h, prices 2c to 98c: {'n': 2203.0, 'events': 928.0, 'slope': 1.012, 'lo': 0.919, 'hi': 1.105, 'brier': 0.123}


,n,events,slope,lo,hi,brier
resolved_at,,,,,,
2026-06,1568.0,403.0,1.014,0.895,1.133,0.047
2026-07,480.0,178.0,0.927,0.713,1.142,0.047
2026-08,824.0,287.0,1.089,0.905,1.272,0.044
2026-09,1771.0,464.0,1.081,0.933,1.229,0.041
2026-10,1162.0,338.0,1.088,0.925,1.250,0.069


## 2. Models vs the market price

Train on events whose markets all closed before Sep 1, test on the rest (an event never sits on both sides).
- Market price: the baseline, price used as the probability
- Logistic recalibration: logit(price), its interaction with horizon and category
- Gradient boosting: price plus 18 features known at the time (24h price change and volatility, order flow toward Yes,
  volume, number of traders, staleness of the last trade, market age, days to scheduled end, event size, category)

Differences in Brier score get event-bootstrap 95% intervals.

In [4]:
eps = 0.005
d['lvwap'] = logit(d.vwap_24h.fillna(d.p_last).clip(eps, 1-eps))
for c in ['dollars_24h', 'dollars_cum', 'n_fills_24h', 'n_wallets_24h', 'n_markets_in_event']:
    d['log_' + c] = np.log1p(d[c].fillna(0))
d['log_age_last'] = np.log1p(d.last_trade_age_h)
d['neg_risk'] = d.neg_risk.astype(float)
for c in ['Politics', 'Technology', 'Entertainment', 'Economy']:
    d['cat_' + c] = (d.category == c).astype(float)
d['lp_x_h'] = d.lp * np.log(d.horizon_h)
for c in ['Technology', 'Entertainment', 'Economy']:
    d['lp_' + c] = d.lp * d['cat_' + c]

recal = ['lp', 'lp_x_h'] + [f'cat_{c}' for c in ['Technology', 'Entertainment', 'Economy']] + [f'lp_{c}' for c in ['Technology', 'Entertainment', 'Economy']]
full = ['lp', 'lvwap', 'horizon_h', 'p_change_24h', 'p_std_24h', 'yes_flow_24h', 'log_dollars_24h', 'log_dollars_cum',
        'log_n_fills_24h', 'log_n_wallets_24h', 'log_age_last', 'market_age_days', 'days_to_scheduled_end', 'neg_risk',
        'log_n_markets_in_event'] + [f'cat_{c}' for c in ['Politics', 'Technology', 'Entertainment', 'Economy']]

SPLIT = pd.Timestamp('2026-09-01', tz='UTC')
test_events = d.groupby('event_slug').resolved_at.max().pipe(lambda x: set(x[x >= SPLIT].index))
d['split'] = np.where(d.event_slug.isin(test_events), 'test', 'train')
tr, te = d[d.split == 'train'], d[d.split == 'test']
print(d.groupby('split').agg(rows=('y', 'size'), markets=('slug', 'nunique'), events=('event_slug', 'nunique')))

preds = {'Market price': te.p_last.clip(1e-4, 1-1e-4).values}
lr = LogisticRegression(C=1.0, max_iter=2000).fit(tr[recal], tr.y)
preds['Logistic recalibration'] = lr.predict_proba(te[recal])[:, 1]
gb = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=15, min_samples_leaf=40,
                                    l2_regularization=1.0, random_state=0).fit(tr[full], tr.y)
preds['Gradient boosting'] = gb.predict_proba(te[full])[:, 1]

        rows  markets  events
split                        
test   10968     3511     860
train  10579     3215     841


In [5]:
def boot_diff(mask, p, B=300):
    ev, y, a = te.event_slug.values[mask], te.y.values[mask], preds['Market price'][mask]
    b = p[mask]; u = np.unique(ev); idx = {e: np.where(ev == e)[0] for e in u}; out = []
    for _ in range(B):
        r = np.concatenate([idx[e] for e in rng.choice(u, len(u))])
        out.append(np.mean((b[r]-y[r])**2) - np.mean((a[r]-y[r])**2))
    return np.percentile(out, [2.5, 97.5])

rows = []
for sub, base in [('all prices', np.ones(len(te), bool)), ('5c to 95c', te.p_last.between(0.05, 0.95, inclusive='neither').values)]:
    for h in [6, 24, 72, 168]:
        m = base & (te.horizon_h == h).values; y = te.y.values[m]
        for k, p in preds.items():
            r = {'subset': sub, 'horizon_h': h, 'model': k, 'n': int(m.sum()), 'brier': np.mean((p[m]-y)**2),
                 'logloss': log_loss(y, np.clip(p[m], 1e-4, 1-1e-4)), 'auc': roc_auc_score(y, p[m])}
            if k != 'Market price':
                r['dBrier_x1000'] = 1000*(r['brier'] - np.mean((preds['Market price'][m]-y)**2))
                lo, hi = boot_diff(m, p); r['ci_lo'], r['ci_hi'] = 1000*lo, 1000*hi
            rows.append(r)
results = pd.DataFrame(rows)
results.round(4)

,subset,horizon_h,model,n,brier,logloss,auc,dBrier_x1000,ci_lo,ci_hi
0,all prices,6,Market price,3052,0.0410,0.1342,0.9834,NaN,NaN,NaN
1,all prices,6,Logistic recalibration,3052,0.0415,0.1367,0.9814,0.4133,-0.0292,0.8532
2,all prices,6,Gradient boosting,3052,0.0415,0.1375,0.9823,0.4461,-1.2930,2.2780
3,all prices,24,Market price,2985,0.0513,0.1703,0.9710,NaN,NaN,NaN
4,all prices,24,Logistic recalibration,2985,0.0518,0.1725,0.9696,0.5275,0.0449,1.0334
5,all prices,24,Gradient boosting,2985,0.0518,0.1744,0.9698,0.5517,-1.5273,3.2922
6,all prices,72,Market price,2865,0.0558,0.1873,0.9620,NaN,NaN,NaN
7,all prices,72,Logistic recalibration,2865,0.0563,0.1897,0.9593,0.4698,0.0141,0.9616
8,all prices,72,Gradient boosting,2865,0.0588,0.1973,0.9577,2.9759,0.3444,5.4404
9,all prices,168,Market price,2066,0.0783,0.2616,0.9226,NaN,NaN,NaN


In [6]:
from sklearn.inspection import permutation_importance
pi = permutation_importance(gb, te[full], te.y, scoring='neg_brier_score', n_repeats=5, random_state=0)
pd.Series(pi.importances_mean, index=full).sort_values(ascending=False).round(5)

lp                        0.10150
lvwap                     0.03990
days_to_scheduled_end     0.00161
neg_risk                  0.00137
market_age_days           0.00059
log_dollars_cum           0.00049
log_n_markets_in_event    0.00026
p_change_24h              0.00024
log_dollars_24h           0.00021
p_std_24h                 0.00017
horizon_h                 0.00013
yes_flow_24h              0.00003
log_age_last              0.00003
cat_Economy               0.00002
cat_Politics              0.00001
log_n_wallets_24h         0.00000
cat_Technology            0.00000
cat_Entertainment         0.00000
log_n_fills_24h          -0.00005
dtype: float64

## Takeaways

1. Prices are well calibrated in all four categories. Within 2c to 98c the calibration slope is about 1.00.
2. The slope above 1 near close (a mild favorite-longshot pattern) comes from the extreme prices and moves from month to month,
   so it is not a stable bias.
3. Accuracy improves as resolution approaches; Economy markets are the hardest to call, Technology and Politics the easiest close to the end.
4. Neither a recalibration nor gradient boosting with 18 extra features beats the price out of sample. The price is almost the only feature that matters.

For the final: the market price is a strong baseline, so an informed trader is one whose bets beat the price, not one who simply wins a lot.

Limitations: price from trades, not the order book (the hourly book covers only about 22% of these markets; where both exist they agree, corr 0.92);
markets that close early because the event happened make "h hours before close" mean different things for Yes and No markets;
one test period (Sep to Oct).